# C5.01 Corpus agenti - Tema 2


Scop: ne uităm la `data/corpus_typed.json`, verificăm distribuția bulelor și alegem ce exemple merită păstrate pentru vector store.



## 1. Setup

In [3]:
from pathlib import Path
import os
import pandas as pd

PROJECT_ROOT = Path(r"D:\ADC\AN 2\18. AI Algorithms\AI_Project\echochamber-project-team-6")
os.chdir(PROJECT_ROOT)
DATA_PATH = Path("data/typed/corpus_typed.jsonl")


In [4]:
DATA_PATH

WindowsPath('data/typed/corpus_typed.jsonl')

## 2. Încărcăm corpusul

In [5]:
df = pd.read_json(DATA_PATH, lines=True)
df.head(2)

,id,source_channel,channel_family,video_title,text,low_information,pre_filtered,target_specific,target_refined,target_l1,...,repr_pluralist_present,repr_pluralist_strength,dem_procedure_rejected,dem_procedure_defended,call_to_action_present,call_to_action_strength,confidence,discourse_type,discourse_subtype,type_confidence
0,yt_iH8jB4NlV9Y_UgxGvCWbzsvB3I-nSy54AaABAg,georgesimionoficial,sovereigntist,Episodul 2: Cum ne-au furat alegerile - Turism...,Metoda votului cetatenilor din Republica Moldo...,False,False,nicusor_dan,nicusor_dan,Instituții stat,...,False,0,False,False,True,1,0.9,T2_grievance_anti_sistem,grievance_conspiratorial,high
1,yt_iH8jB4NlV9Y_UgwiYbOpe89FfXbeIpl4AaABAg,georgesimionoficial,sovereigntist,Episodul 2: Cum ne-au furat alegerile - Turism...,Doamne cu atâtea dovezi și nicio instituție in...,False,False,simion,simion,Sovereigntist,...,False,0,False,False,False,0,0.9,T1_suport_personalist,suport_afectiv_suveranist,medium


## 3. Vedem structura datelor

In [6]:
len(df)

17886

In [7]:
print("Coloane:")
df.columns

Coloane:


Index(['id', 'source_channel', 'channel_family', 'video_title', 'text',
       'low_information', 'pre_filtered', 'target_specific', 'target_refined',
       'target_l1', 'target_l2', 'stance_to_target', 'primary_target_hint',
       'target_confidence', 'inst_neg_present', 'inst_neg_strength',
       'inst_pos_present', 'inst_pos_strength',
       'epist_hidden_coordination_present',
       'epist_hidden_coordination_strength',
       'epist_evidence_verification_present',
       'epist_evidence_verification_strength',
       'geo_anti_external_domination_present',
       'geo_anti_external_domination_strength',
       'geo_pro_external_anchoring_present',
       'geo_pro_external_anchoring_strength', 'repr_personalist_present',
       'repr_personalist_strength', 'repr_pluralist_present',
       'repr_pluralist_strength', 'dem_procedure_rejected',
       'dem_procedure_defended', 'call_to_action_present',
       'call_to_action_strength', 'confidence', 'discourse_type',
       'disco

In [8]:
df["discourse_type"].value_counts(dropna=False)

discourse_type
T2_grievance_anti_sistem      7583
T1_suport_personalist         4624
T6_afectiv_pozitional         3726
T4_conspiratie_externalism    1506
T3_opozitie_suveranista        261
T5_pro_democratic_european     186
Name: count, dtype: int64

In [26]:
## Exemplu de text pentru fiecare tip de discurs

In [9]:
for bubble in df["discourse_type"].value_counts().index:
    print("\n" + "="*80)
    print(bubble)
    print("="*80)
    
    sample = df[df["discourse_type"] == bubble]["text"].dropna().head(5)
    
    for i, text in enumerate(sample, 1):
        print(f"\n{i}. {text[:50]}")


T2_grievance_anti_sistem

1. Metoda votului cetatenilor din Republica Moldova a

2. Calin Georgescu si George Simion la puscarieeee!! 

3. S-au furat prin liste suplimentare, într-un sat di

4. Eu nu cred că caracatița poate fi dată la parte !E

5. Știu ca există un sistem de vot electronic în care

T1_suport_personalist

1. Doamne cu atâtea dovezi și nicio instituție intern

2. Noi oricum știm asta dar întrebarea mea este domnu

3. Felicitări și RESPECT George Simion! Nu te opri! V

4. Haideți in stradă pe 20 iulie!!!! Să dăm jos hoții

5. MILIOANE DE ROMÂNI DEMNI SUNTEM ALĂTURI DE TINE,GE

T6_afectiv_pozitional

1. Adevarul sa fie facut prezent si rezultatul sa fie

2. Ceea ce face este demn de noi romanii adevarati . 

3. Si de trebuie deposite de combustibil degeaba avem

4. Dumnezeu sa Binecuvinteze Partidul Aur, 🇹🇩🙏 Români

5. Felicitari pentru discurs! ❤️🙏🇷🇴 Si pentru consecv

T4_conspiratie_externalism

1. Atunci daca aveți dovezi împotriva la Maya Sandu d

2. Dom-le am văzut 

# 4. Pastram doar 150 de bula

In [10]:
# Alegem un subset simplu pentru curs: maximum 150 texte per bulă

N_PER_BUBBLE = 150

df_sample = (
    df[df["discourse_type"].notna()]
    .sort_values("type_confidence", ascending=False)
    .groupby("discourse_type", group_keys=False)
    .head(N_PER_BUBBLE)
    .copy()
)

df_sample["discourse_type"].value_counts()

discourse_type
T3_opozitie_suveranista       150
T2_grievance_anti_sistem      150
T1_suport_personalist         150
T4_conspiratie_externalism    150
T5_pro_democratic_european    150
T6_afectiv_pozitional         150
Name: count, dtype: int64

In [11]:
# Păstrăm doar coloanele utile și salvăm corpusul pentru etapa următoare

keep_cols = [
    "id", "text", "source_channel", "channel_family", "video_title",
    "target_refined", "stance_to_target",
    "confidence", "discourse_type", "discourse_subtype", "type_confidence"
]

df_sample = df_sample[keep_cols].drop_duplicates(subset="text").copy()

OUT_PATH = Path("data/typed/corpus_c5_sample.jsonl")
df_sample.to_json(OUT_PATH, orient="records", lines=True, force_ascii=False)

print(df_sample.shape)
print(OUT_PATH)
df_sample.head(2)

(896, 11)
data\typed\corpus_c5_sample.jsonl


,id,text,source_channel,channel_family,video_title,target_refined,stance_to_target,confidence,discourse_type,discourse_subtype,type_confidence
17885,yt_Tx8GhU2LeyI_UgwoWOyzF2UbPYnguUB4AaABAg,Am toată încrederea că oameni ( de bine ) ca :...,AlephNewsOfficial,mainstream,ATENȚIE: România e „binevenită” să aplice iar ...,simion,anti,0.9,T3_opozitie_suveranista,opozitie_difuza,medium
15394,yt_joXkZDqGZQU_Ugyqb1XZ7P8GTnJS_4p4AaABAg,Semneaza Bo$$ ca la urmatoarele alegerii nu ma...,NicusorDanRO,mainstream_actor,🟢 Declarații de presă comune cu Președintele U...,nicusor_dan,anti,0.9,T2_grievance_anti_sistem,grievance_mobilizator,medium


## 5. Exportul bulelor finale
În această etapă transformăm corpusul tipologizat în fișiere separate, câte unul pentru fiecare bulă finală.
Codurile `T1`, `T2`, `T3`, `T4`, `T5` vin din adnotare, dar în aplicație folosim numele finale ale agenților:
| Agent | Personalitate | Cum vorbește | Ce îl definește |
|---|---|---|---|
| Personalist-salvator | devotat, admirativ, sigur | laudativ, emoțional, încrezător | vede liderul ca soluție excepțională |
| Anti-sistem | furios, suspicios, dezamăgit | acuzator, moralizator, direct | vede instituțiile și „sistemul” ca profund compromise |
| Anti-suveranist | critic, vigilent, defensiv | contestatar, mai argumentativ | respinge liderii și discursul suveranist |
| Conspiraționist | alarmist, hiper-suspicios | speculativ, revelator, totalizant | explică evenimentele prin forțe ascunse și actori externi |
| Pro-european | normativ, moderat, legalist | sobru, justificativ, procedural | apără regulile, instituțiile și ancorarea europeană |
`T6_afectiv_pozitional` nu devine agent principal în C5. Rămâne categorie transversală / rezervă.

In [12]:
BUBBLES = {
    "T1_suport_personalist": {
        "agent": "Personalist-salvator",
        "slug": "personalist_salvator",
        "personality": "devotat, admirativ, sigur",
        "speaks": "laudativ, emoțional, încrezător",
        "definition": "vede liderul ca soluție excepțională",
    },
    "T2_grievance_anti_sistem": {
        "agent": "Anti-sistem",
        "slug": "anti_sistem",
        "personality": "furios, suspicios, dezamăgit",
        "speaks": "acuzator, moralizator, direct",
        "definition": "vede instituțiile și „sistemul” ca profund compromise",
    },
    "T3_opozitie_suveranista": {
        "agent": "Anti-suveranist",
        "slug": "anti_suveranist",
        "personality": "critic, vigilent, defensiv",
        "speaks": "contestatar, mai argumentativ",
        "definition": "respinge liderii și discursul suveranist",
    },
    "T4_conspiratie_externalism": {
        "agent": "Conspiraționist",
        "slug": "conspirationist",
        "personality": "alarmist, hiper-suspicios",
        "speaks": "speculativ, revelator, totalizant",
        "definition": "explică evenimentele prin forțe ascunse și actori externi",
    },
    "T5_pro_democratic_european": {
        "agent": "Pro-european",
        "slug": "pro_european",
        "personality": "normativ, moderat, legalist",
        "speaks": "sobru, justificativ, procedural",
        "definition": "apără regulile, instituțiile și ancorarea europeană",
    },
}

In [13]:
"""
OUT_DIR = Path("data/bubbles")
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_PER_BUBBLE = 50

for old_type, meta in BUBBLES.items():
    bubble_df = (
        df_sample[df_sample["discourse_type"] == old_type]
        .drop_duplicates(subset="text")
        .head(N_PER_BUBBLE)
        .copy()
    )

    bubble_df["agent"] = meta["agent"]
    bubble_df["slug"] = meta["slug"]
    bubble_df["personality"] = meta["personality"]
    bubble_df["speaks"] = meta["speaks"]
    bubble_df["definition"] = meta["definition"]
    bubble_df["source_type"] = old_type

    out_path = OUT_DIR / f"{meta['slug']}.jsonl"
    bubble_df.to_json(out_path, orient="records", lines=True, force_ascii=False)

    print(f"{meta['agent']}: {len(bubble_df)} texte -> {out_path}")

"""

'\nOUT_DIR = Path("data/bubbles")\nOUT_DIR.mkdir(parents=True, exist_ok=True)\n\nN_PER_BUBBLE = 50\n\nfor old_type, meta in BUBBLES.items():\n    bubble_df = (\n        df_sample[df_sample["discourse_type"] == old_type]\n        .drop_duplicates(subset="text")\n        .head(N_PER_BUBBLE)\n        .copy()\n    )\n\n    bubble_df["agent"] = meta["agent"]\n    bubble_df["slug"] = meta["slug"]\n    bubble_df["personality"] = meta["personality"]\n    bubble_df["speaks"] = meta["speaks"]\n    bubble_df["definition"] = meta["definition"]\n    bubble_df["source_type"] = old_type\n\n    out_path = OUT_DIR / f"{meta[\'slug\']}.jsonl"\n    bubble_df.to_json(out_path, orient="records", lines=True, force_ascii=False)\n\n    print(f"{meta[\'agent\']}: {len(bubble_df)} texte -> {out_path}")\n\n'

## 6. Alege agentul tău și verifică textele
Fiecare membru al echipei lucrează pe un singur agent.
Datele vin din `df_sample`, iar agentul este legat de eticheta tehnică `discourse_type`.
Scopul este să alegi aproximativ 50 texte bune pentru agentul tău.

In [14]:
# Alegeți un agent și încărcați textele corespunzătoare pentru etapa următoare
AGENTS = {
    "Personalist-salvator": {
        "type": "T1_suport_personalist",
        "slug": "personalist_salvator",
        "personality": "devotat, admirativ, sigur",
        "speaks": "laudativ, emoțional, încrezător",
        "definition": "vede liderul ca soluție excepțională",
    },
    "Anti-sistem": {
        "type": "T2_grievance_anti_sistem",
        "slug": "anti_sistem",
        "personality": "furios, suspicios, dezamăgit",
        "speaks": "acuzator, moralizator, direct",
        "definition": "vede instituțiile și „sistemul” ca profund compromise",
    },
    "Anti-suveranist": {
        "type": "T3_opozitie_suveranista",
        "slug": "anti_suveranist",
        "personality": "critic, vigilent, defensiv",
        "speaks": "contestatar, mai argumentativ",
        "definition": "respinge liderii și discursul suveranist",
    },
    "Conspiraționist": {
        "type": "T4_conspiratie_externalism",
        "slug": "conspirationist",
        "personality": "alarmist, hiper-suspicios",
        "speaks": "speculativ, revelator, totalizant",
        "definition": "explică evenimentele prin forțe ascunse și actori externi",
    },
    "Pro-european": {
        "type": "T5_pro_democratic_european",
        "slug": "pro_european",
        "personality": "normativ, moderat, legalist",
        "speaks": "sobru, justificativ, procedural",
        "definition": "apără regulile, instituțiile și ancorarea europeană",
    },
}

MY_AGENT = "Pro-european"  # Alegeți unul dintre agenți: "Personalist-salvator", "Anti-sistem", "Anti-suveranist", "Conspiraționist", "Pro-european"

meta = AGENTS[MY_AGENT]

my_df = (
    df_sample[df_sample["discourse_type"] == meta["type"]]
    .drop_duplicates(subset="text")
    .copy()
)

print("Agent:", MY_AGENT)
print("Tip discurs:", meta["type"])
print("Texte disponibile:", len(my_df))

my_df[["id", "type_confidence", "discourse_subtype", "text"]].head(2)

Agent: Pro-european
Tip discurs: T5_pro_democratic_european
Texte disponibile: 150


,id,type_confidence,discourse_subtype,text
15478,yt_6_Hc2S02Duw_UgwTw7_YpNZEUGkYDb94AaABAg,medium,legitimitate_pluralista,"Nu are Ce cauta pe teritoriulRomaniei, indifer..."
15638,yt_yEuctxNb4O0_UgxTCkwcP96Sb5_Rpht4AaABAg,medium,legitimitate_pluralista,Tipul care și-a dat demisia în noiembrie la cu...


### Cum verifici textele
Citește textele afișate mai jos.
Dacă un text este slab, copiază ID-ul lui în lista `REMOVE_IDS`.
Elimină texte prea scurte, duplicate, ambigue sau care nu exprimă clar vocea agentului.|

- pot sa folosesc un notepad pentru IDs
daca nu gasesti 50 de comentarii bune, incarca mai multe.
- poti folosi si alte metode (export text, csv) pentru vizualizarea si alegerea comentariilor

In [16]:
for _, row in my_df.head(70).iterrows():
    print("=" * 80)
    print("ID:", row["id"])
    print("Confidence:", row["type_confidence"])
    print("Subtype:", row["discourse_subtype"])
    print(row["text"][:700])

ID: yt_6_Hc2S02Duw_UgwTw7_YpNZEUGkYDb94AaABAg
Confidence: medium
Subtype: legitimitate_pluralista
Nu are Ce cauta pe teritoriulRomaniei, indiferent ca are s-au nu are , ca sint s-au ca nu sint defensive, s-au offensive- nu au acordul poporului
ID: yt_yEuctxNb4O0_UgxTCkwcP96Sb5_Rpht4AaABAg
Confidence: medium
Subtype: legitimitate_pluralista
Tipul care și-a dat demisia în noiembrie la cum gândește și vorbește merita sa fie ministrul justiției. Oameni tineri, capabili, încă necorupți, fără trecut securist/comunist trebuie sa fie în funcții cheie ale statului. Vezi miniștrii USR, maxim 40 de ani, pregătiți, implicați....
ID: yt_kLdQS9N5cBU_UgxIxee_zETaR-Dlu894AaABAg
Confidence: medium
Subtype: legitimitate_pluralista
+ la asta, d. Președinte, trebuie sa existe O MAJORITATE a cetatenilor si in Romănia..., care sa doreasca UNIREA....
ID: yt_FAe_Sqr2vfU_UgzfkQjlhFS0o3ZaCMl4AaABAg
Confidence: medium
Subtype: legitimitate_pluralista
Mă bucur să văd un Președinte care se comportă firesc și zâmbe

In [19]:
# elimin textele slabe și păstrez 50

REMOVE_IDS = [
    "yt_6_Hc2S02Duw_UgwTw7_YpNZEUGkYDb94AaABAg"
]

clean_df = (
    my_df[~my_df["id"].isin(REMOVE_IDS)]
    .head(50)
    .copy()
)

clean_df["agent"] = MY_AGENT
clean_df["slug"] = meta["slug"]
clean_df["personality"] = meta["personality"]
clean_df["speaks"] = meta["speaks"]
clean_df["definition"] = meta["definition"]

print("Texte finale:", len(clean_df))
clean_df[["id", "agent", "text"]].head()

Texte finale: 50


,id,agent,text
15638,yt_yEuctxNb4O0_UgxTCkwcP96Sb5_Rpht4AaABAg,Pro-european,Tipul care și-a dat demisia în noiembrie la cu...
15511,yt_kLdQS9N5cBU_UgxIxee_zETaR-Dlu894AaABAg,Pro-european,"+ la asta, d. Președinte, trebuie sa existe O ..."
16422,yt_FAe_Sqr2vfU_UgzfkQjlhFS0o3ZaCMl4AaABAg,Pro-european,Mă bucur să văd un Președinte care se comportă...
17650,yt_Fcmw2yFzz8I_UgxuNvJZTDz3pFm5Ijh4AaABAg,Pro-european,Deci exemplu asta cu Parisul cred ca e cea mai...
16916,yt_XlCX1l_6feU_UgwBIpp5ZbEtJkybe3t4AaABAg,Pro-european,"meritocratie in politica inseamna, totusi, alt..."


### Descrierea agentului tău
Înainte să exporți bula curată, descrie în 5–7 rânduri ce fel de voce discursivă are agentul ales.
Nu descrie o persoană reală. Descrie un tip de discurs observat în corpus.
Răspunde la aceste întrebări:
- Cum vede acest agent instituțiile, politica sau actorii publici?
- Ce ton folosește cel mai des?
- Ce tip de argumente sau acuzații apar frecvent?
- Ce îl diferențiază de celelalte bule?
- Ce ar trebui să păstreze un viitor agent AI ca să sune coerent cu această bulă?

descriere:
Acest agent vede instituțiile ca fiind coloana vertebrală a societății, tratând politica nu ca pe un spectacol, ci ca pe un exercițiu riguros de administrare a binelui comun prin respectarea regulilor. Perspectiva sa este una tehnocratică: actorii publici sunt validați doar prin competență și integritate procedurală.

Tonul utilizat este invariabil sobru, echilibrat și adesea didactic, evitând stridențele emoționale în favoarea clarității juridice. Argumentele sale gravitează constant în jurul statului de drept și al ancorării europene, acuzând „derapajele” populiste sau „amatorismul” celor care ignoră normele. Ceea ce îl diferențiază radical de bulele suveraniste sau conspiraționiste este încrederea neclintită în mecanismele de control și în tratatele internaționale, respingând ideea de „salvator”.

Pentru a păstra coerența acestei bule, un viitor agent AI trebuie să adopte un limbaj precis, aproape academic, care să prioritizeze sursele oficiale și rigoarea legislativă. Trebuie să rămână o voce a stabilității care, în loc de soluții utopice, oferă analize de impact și argumente bazate pe predictibilitate și continuitate instituțională.

In [20]:
# export bula curată pentru etapa următoare

OUT_DIR = Path("data/bubbles")
OUT_DIR.mkdir(parents=True, exist_ok=True)

out_path = OUT_DIR / f"{meta['slug']}.jsonl"

clean_df.to_json(out_path, orient="records", lines=True, force_ascii=False)

print("Salvat:", out_path)

Salvat: data\bubbles\pro_european.jsonl
